# Credit Card Fraud Detection. Exploratory Data Analysis

**Project:** ML Fraud Detection Pipeline · **Author:** Felipe Toro · **Notebook:** 01 of 03

---

## Purpose

Establish the data, the schema, and the cost framing that drive every subsequent modeling decision. The output of this notebook is a set of modeling decisions. not a model.

## The modeling problem in one paragraph

Credit card fraud detection presents three structural challenges that together rule out conventional classification approaches: severe class imbalance (under 0.2% fraud rate), asymmetric error costs (a missed fraud costs ~25x more than a false alarm), and anonymized PCA-transformed features that prevent classical feature engineering. This notebook quantifies all three precisely enough to drive the design of the training pipeline in Notebook 02.

---

## Notebook structure

1. Setup and data loading
2. Schema and data integrity
3. Class imbalance and cost framing
4. Univariate distributions
5. Feature relationships with the target
6. Time and transaction amount (the only non-anonymized features)
7. Correlation structure
8. Key findings and modeling decisions

---
## 1. Setup and data loading

Imports and the project's centralized data loader. Path detection anchors on `README.md` as a marker file so the notebook executes correctly from any working directory.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Project root resolves relative to README.md location, not the working
# directory. This makes the notebook portable across containers, CI runs,
# and arbitrary launch contexts without path manipulation at call sites.
_cwd = Path.cwd()
if (_cwd / "README.md").exists():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "README.md").exists():
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd

sys.path.insert(0, str(PROJECT_ROOT))

# Output directory for publication-quality figures referenced by the README
# and the case study page. Created idempotently so re-runs do not fail.
IMAGES_DIR = PROJECT_ROOT / "docs" / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Loader exports the single source of truth for schema constants. Importing
# from here rather than redefining locally guarantees the EDA cannot drift
# from the training pipeline's view of the data.
from src.data.loader import (
    load_training_data,
    PCA_FEATURES, AMOUNT_COL, TIME_COL, TARGET_COL,
)

# Plotting configuration. The fixed dpi values produce consistent figure
# resolution across machines, which matters because the saved PNGs are
# committed to the repo and referenced in documentation.
sns.set_theme(style="whitegrid", palette="muted", context="notebook")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 12

# Color palette aligned with the portfolio site so figures embedded in the
# case study match the surrounding design without per-image restyling.
COLOR_BG = "#0d1117"
COLOR_FRAUD = "#ef4444"
COLOR_LEGIT = "#4f8ef7"
COLOR_NEUTRAL = "#8a9bbf"

print("Imports successful.")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy:  {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Load the full labeled training dataset. The loader raises FileNotFoundError
# at this point if the CSV is missing, surfacing the issue immediately rather
# than producing empty downstream frames.
df = load_training_data()
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

---
## 2. Schema and data integrity

Column types, missingness, and duplicates are checked before any analysis. Catching schema issues here prevents the kind of silent failure mode where a typo in a column name produces NaN-filled metrics later.

In [ ]:
# Schema snapshot: dtype, cardinality, and missingness per column. Anything
# outside expected ranges (e.g. non-numeric dtypes on PCA features) is a
# signal that the data has changed shape since the pipeline was last tested.
schema_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "n_missing": df.isnull().sum(),
    "pct_missing": (df.isnull().sum() / len(df) * 100).round(3),
})
schema_summary.head(15)

In [ ]:
# Aggregate missingness check. The training set is expected to have minor
# missingness (~61 cells out of millions); anything substantially larger
# indicates either a load error or upstream data corruption.
total_missing = df.isnull().sum().sum()
print(f"Total missing values: {total_missing:,} ({total_missing / df.size * 100:.4f}% of cells)")
print(f"Columns with missing values: {(df.isnull().sum() > 0).sum()}")
print()
if total_missing > 0:
    print("Per-column missingness (non-zero only):")
    print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Duplicate detection on both full-row identity and Transaction ID. A
# non-zero ID duplicate count would indicate the source data has been
# concatenated incorrectly upstream, which would inflate any frequency-
# based statistic computed afterward.
n_duplicate_rows = df.duplicated().sum()
n_duplicate_ids = df["Transaction"].duplicated().sum()
print(f"Fully duplicate rows: {n_duplicate_rows:,}")
print(f"Duplicate Transaction IDs: {n_duplicate_ids:,}")

**Integrity check: passed.** Minor missingness (61 cells of ~7.5M) is handled by median imputation inside the sklearn Pipeline in Notebook 02. No duplicate transactions in the source data.

---
## 3. Class imbalance and cost framing

The single most important section of the EDA. Class distribution and cost asymmetry together shape every modeling decision in Notebook 02. the choice of CV scoring metric, the threshold optimization strategy, and ultimately the model selection criterion.

In [ ]:
# Target distribution. The imbalance ratio is the headline number: it
# defines the upper bound on what accuracy means and tells the modeling
# pipeline that stratified splits and class-aware metrics are non-negotiable.
target_counts = df[TARGET_COL].value_counts().sort_index()
fraud_rate = df[TARGET_COL].mean()

print(f"Legitimate transactions: {target_counts[0]:,}  ({(1 - fraud_rate) * 100:.4f}%)")
print(f"Fraudulent transactions: {target_counts[1]:,}  ({fraud_rate * 100:.4f}%)")
print(f"Class imbalance ratio:   1 : {target_counts[0] / target_counts[1]:,.0f}")

In [ ]:
# Dual-scale visualization. Linear scale conveys the magnitude of the
# imbalance viscerally; log scale confirms the minority class is non-zero
# and inspectable. Showing both heads off the misreading that fraud is
# absent from the data.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

bar_colors = [COLOR_LEGIT, COLOR_FRAUD]
labels = ["Legitimate", "Fraud"]

axes[0].bar(labels, target_counts.values, color=bar_colors, edgecolor="white", linewidth=2)
axes[0].set_title("Class Distribution (Linear Scale)", fontsize=12, pad=12)
axes[0].set_ylabel("Number of transactions")
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v, f"  {v:,}", va="bottom", ha="center", fontweight="bold")

axes[1].bar(labels, target_counts.values, color=bar_colors, edgecolor="white", linewidth=2)
axes[1].set_yscale("log")
axes[1].set_title("Class Distribution (Log Scale)", fontsize=12, pad=12)
axes[1].set_ylabel("Number of transactions (log)")
for i, v in enumerate(target_counts.values):
    axes[1].text(i, v, f"  {v:,}", va="bottom", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(IMAGES_DIR / "01_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### Accuracy is the wrong metric here

A classifier that labels every transaction as legitimate achieves:

- Accuracy: 99.82%
- Recall on fraud: 0%
- Real-world value: zero

The training pipeline therefore evaluates on **precision, recall, F1, AUC-PR, and a custom cost-weighted score**. never on accuracy. Optuna's CV objective uses AUC-PR (average precision) because it is the right metric for severe imbalance: it summarizes precision across all recall thresholds without being skewed by the dominant negative class.

In [ ]:
# Dollar value at risk. The mean fraud transaction value is the FN cost
# that drives the CostMatrix in src/utils/cost_metrics.py. these numbers
# are not abstractions, they are the inputs to the optimizer that selects
# the production threshold.
fraud_amounts = df.loc[df[TARGET_COL] == 1, AMOUNT_COL]
legit_amounts = df.loc[df[TARGET_COL] == 0, AMOUNT_COL]

cost_summary = pd.DataFrame({
    "Metric": ["Count", "Total $", "Mean $", "Median $"],
    "Legitimate": [
        f"{int(legit_amounts.count()):,}",
        f"${legit_amounts.sum():,.2f}",
        f"${legit_amounts.mean():,.2f}",
        f"${legit_amounts.median():,.2f}",
    ],
    "Fraud": [
        f"{int(fraud_amounts.count()):,}",
        f"${fraud_amounts.sum():,.2f}",
        f"${fraud_amounts.mean():,.2f}",
        f"${fraud_amounts.median():,.2f}",
    ],
}).set_index("Metric")

print("Total dollar value at risk in the training data:")
print(cost_summary.to_string())

In [ ]:
# Cost matrix derivation. The FN cost is the empirical mean fraud
# transaction value; the FP cost is the industry-standard manual review
# cost per flagged transaction. The 25:1 ratio is what makes naive
# threshold selection at 0.5 catastrophically suboptimal. at that
# threshold, the model trades cheap reviews for expensive missed fraud
# in the wrong direction.
mean_fraud_amount = fraud_amounts.mean()
fp_review_cost = 5.00

print("Cost matrix used in threshold optimization (Notebook 02):")
print(f"  False Negative cost (missed fraud):   ${mean_fraud_amount:,.2f}  (avg fraud transaction)")
print(f"  False Positive cost (false alarm):    ${fp_review_cost:.2f}    (manual review)")
print(f"  Cost ratio (FN / FP):                 {mean_fraud_amount / fp_review_cost:.1f}x")
print()
print(f"Implication: catching fraud is ~{mean_fraud_amount / fp_review_cost:.0f}x more valuable than")
print("avoiding false alarms. Threshold tuning therefore dominates F1 maximization.")

---
## 4. Univariate distributions

Visual inspection of PCA-transformed predictors. Since `X1`–`X28` are PCA components, the marginal distributions are expected to be approximately mean-zero with varying scales. The relevant question is not whether the features are normally distributed (they are not, by design) but whether the fraud and legitimate subpopulations differ enough on any single feature to be useful predictors.

In [ ]:
# Distribution overlay for the first 12 PCA components. Visualizing all 28
# would produce illegible subplots; the first 12 cover the highest-variance
# components and are sufficient to read the separability pattern.
#
# Legitimate distribution is downsampled to 20k rows for plotting speed.
# Fraud distribution is plotted in full (only 145 rows exist).
features_to_plot = PCA_FEATURES[:12]

fig, axes = plt.subplots(3, 4, figsize=(16, 9))
axes = axes.ravel()

for i, feat in enumerate(features_to_plot):
    ax = axes[i]
    legit_sample = df[df[TARGET_COL] == 0][feat].sample(20000, random_state=42)
    fraud_all = df[df[TARGET_COL] == 1][feat]

    ax.hist(legit_sample, bins=50, density=True, alpha=0.55,
            color=COLOR_LEGIT, label="Legitimate", edgecolor="white", linewidth=0.4)
    ax.hist(fraud_all, bins=30, density=True, alpha=0.75,
            color=COLOR_FRAUD, label="Fraud", edgecolor="white", linewidth=0.4)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel("")
    ax.set_yticks([])
    if i == 0:
        ax.legend(loc="upper right", fontsize=8, framealpha=0.9)

plt.suptitle("Distribution of PCA Features X1-X12: Fraud vs Legitimate",
             fontsize=14, fontweight="bold", y=1.00)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "02_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation:** Several PCA components show clear class separation. `X3`, `X4`, `X7`, `X10`, `X11`, `X12` in particular. Others overlap heavily and carry little univariate signal. Importantly, no single feature cleanly partitions the two classes: the decision boundary is non-linear in feature space. This is the diagnostic that justifies gradient-boosted trees over linear models in Notebook 02. tree splits capture interactions that a linear decision boundary cannot.

---
## 5. Feature relationships with the target

Welch's t-test per feature, comparing the means of the fraud and legitimate distributions. The absolute t-statistic ranks features by univariate discriminative power. This ranking is a screening tool, not a final feature-importance estimate. the model's SHAP values in Notebook 02 are the authoritative attribution because they account for feature interactions.

In [ ]:
# Welch's t-test handles unequal variances between the two distributions,
# which is the more defensible assumption for fraud vs legitimate (the
# fraud distribution is much smaller and likely has different variance).
from scipy.stats import ttest_ind

separability = []
for feat in PCA_FEATURES + [AMOUNT_COL, TIME_COL]:
    fraud_vals = df.loc[df[TARGET_COL] == 1, feat].dropna()
    legit_vals = df.loc[df[TARGET_COL] == 0, feat].dropna()
    t_stat, p_val = ttest_ind(fraud_vals, legit_vals, equal_var=False)
    separability.append({
        "feature": feat,
        "fraud_mean": fraud_vals.mean(),
        "legit_mean": legit_vals.mean(),
        "abs_t_stat": abs(t_stat),
        "p_value": p_val,
    })

sep_df = pd.DataFrame(separability).sort_values("abs_t_stat", ascending=False)
sep_df.head(10).round(4)

In [ ]:
# Visualization of the top 10 features by |t-statistic|. The top 3 are
# highlighted in the fraud color to emphasize the most-discriminative
# predictors. these are the features the production model is expected
# to weight most heavily, and SHAP will confirm or refute this in
# Notebook 02.
top_features = sep_df.head(10)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(top_features["feature"][::-1], top_features["abs_t_stat"][::-1],
               color=COLOR_LEGIT, edgecolor="white", linewidth=1.2)

for i, bar in enumerate(bars[-3:]):
    bar.set_color(COLOR_FRAUD)

ax.set_xlabel("|t-statistic|  (higher = stronger fraud / legitimate separation)")
ax.set_title("Top 10 Most Discriminative Features (Welch's t-test)", fontsize=13, pad=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "03_top_features.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Time and transaction amount

`X29` (transaction amount) and `X30` (seconds elapsed since first transaction) are the only non-anonymized features. They deserve dedicated treatment because they are the only places where domain knowledge can drive feature engineering. Everything else is a PCA component with no business interpretation.

In [ ]:
# Transaction amount: distribution comparison.
#
# Log scale on the x-axis is necessary because transaction amounts are
# heavy-tailed. a small number of high-value transactions would compress
# the visible range otherwise. The boxplot hides outliers because they
# obscure the central tendency comparison.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for label, color in [(0, COLOR_LEGIT), (1, COLOR_FRAUD)]:
    subset = df.loc[df[TARGET_COL] == label, AMOUNT_COL]
    axes[0].hist(np.log1p(subset), bins=60, density=True, alpha=0.65,
                 color=color, label="Fraud" if label == 1 else "Legitimate",
                 edgecolor="white", linewidth=0.4)
axes[0].set_xlabel("log(1 + Transaction Amount $)")
axes[0].set_ylabel("Density")
axes[0].set_title("Transaction Amount Distribution (log scale)")
axes[0].legend()

data_to_plot = [
    df.loc[df[TARGET_COL] == 0, AMOUNT_COL],
    df.loc[df[TARGET_COL] == 1, AMOUNT_COL],
]
bp = axes[1].boxplot(data_to_plot, labels=["Legitimate", "Fraud"],
                     patch_artist=True, showfliers=False, widths=0.55)
for patch, color in zip(bp["boxes"], [COLOR_LEGIT, COLOR_FRAUD]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1].set_ylabel("Transaction Amount $")
axes[1].set_title("Amount by Class (outliers hidden)")

plt.tight_layout()
plt.savefig(IMAGES_DIR / "04_amount_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nLegitimate amount . median: ${df.loc[df[TARGET_COL] == 0, AMOUNT_COL].median():.2f},  "
      f"max: ${df.loc[df[TARGET_COL] == 0, AMOUNT_COL].max():,.2f}")
print(f"Fraud amount      . median: ${df.loc[df[TARGET_COL] == 1, AMOUNT_COL].median():.2f},  "
      f"max: ${df.loc[df[TARGET_COL] == 1, AMOUNT_COL].max():,.2f}")

In [ ]:
# Time distribution by class. Seconds are converted to hours since the
# first transaction to make the temporal pattern legible. Plotting fraud
# and legitimate on separate y-axes (rather than overlaid) is necessary
# because the volume difference would otherwise hide the fraud pattern
# entirely.
df["hour_since_start"] = df[TIME_COL] / 3600.0

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

legit_by_hour = df.loc[df[TARGET_COL] == 0, "hour_since_start"]
axes[0].hist(legit_by_hour, bins=80, color=COLOR_LEGIT, alpha=0.85, edgecolor="white")
axes[0].set_ylabel("Legitimate transactions per bin")
axes[0].set_title("Transaction Volume Over Time: by Class", fontsize=12, pad=8)

fraud_by_hour = df.loc[df[TARGET_COL] == 1, "hour_since_start"]
axes[1].hist(fraud_by_hour, bins=80, color=COLOR_FRAUD, alpha=0.85, edgecolor="white")
axes[1].set_ylabel("Fraud transactions per bin")
axes[1].set_xlabel("Hours since first transaction")

plt.tight_layout()
plt.savefig(IMAGES_DIR / "05_time_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# Drop the helper column so downstream code sees the original schema.
df = df.drop(columns=["hour_since_start"])

**Observation:** Legitimate transactions show a clear daily volume cycle consistent with typical online activity patterns. Fraud transactions do not follow the same rhythm. This temporal divergence is signal. time-of-day or time-since-last-transaction features could carry predictive power. The training pipeline in Notebook 02 leaves these features in their raw form so the tree-based models can discover the interactions natively, rather than baking in a manually-engineered feature that may underfit the actual pattern.

---
## 7. Correlation structure

The PCA components should be approximately uncorrelated by construction. Verification matters: a non-trivial correlation between features that are supposed to be orthogonal would indicate either an upstream data issue or that the PCA basis has been transformed since publication.

In [ ]:
# Pearson correlation of each predictor with the target. This is a linear
# correlation only. it captures the strength of the linear component of
# the relationship and ignores everything else. Features with low Pearson
# correlation can still be highly predictive through tree splits, which is
# part of why the next notebook emphasizes tree-based models.
corr_with_target = df[PCA_FEATURES + [AMOUNT_COL, TIME_COL] + [TARGET_COL]].corr()[TARGET_COL]
corr_with_target = corr_with_target.drop(TARGET_COL).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(11, 7))
colors = [COLOR_FRAUD if v > 0 else COLOR_LEGIT for v in corr_with_target.values]
ax.barh(corr_with_target.index[::-1], corr_with_target.values[::-1],
        color=colors[::-1], edgecolor="white", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.7)
ax.set_xlabel("Pearson correlation with fraud label")
ax.set_title("Linear Correlation with Target: Ranked", fontsize=13, pad=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "06_correlation_with_target.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Pairwise correlation among PCA predictors. A near-diagonal heatmap
# confirms the PCA basis is orthogonal as expected. Any off-diagonal
# colored cell would warrant investigation before trusting downstream
# multivariate analysis.
pca_corr = df[PCA_FEATURES].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pca_corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.7}, linewidths=0.3, ax=ax)
ax.set_title("Correlation Matrix: PCA Components X1-X28", fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "07_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation:** The PCA components are essentially uncorrelated with each other (as expected from a PCA transformation), so multicollinearity is not a concern for any model in the candidate set. Pairwise correlations between `X29` (amount) and the PCA components are also weak, meaning transaction amount carries somewhat independent signal. useful for the tree-based models that can combine it with the PCA components through splits.

---
## 8. Key findings and modeling decisions

This is the bridge between EDA and the training pipeline. Each finding below maps directly to a decision encoded in Notebook 02.

### Findings

1. **Severe class imbalance: 0.183% fraud rate (1 fraud per ~545 transactions).**
   Drives the choice of AUC-PR as the CV scoring metric, stratified k-fold splits, and class-aware loss weighting where supported.

2. **Asymmetric error costs: FN $125.17 vs FP $5.00 (25:1 ratio).**
   Drives cost-sensitive threshold optimization. The production threshold is selected by minimizing expected dollar cost across a 200-point sweep, not by maximizing F1 or accuracy.

3. **Non-linear feature separation.** Several PCA features (X3, X4, X7, X10, X11, X12, X14, X17) show clear class separation but no single feature cleanly partitions the classes.
   Drives the inclusion of gradient-boosted ensembles (XGBoost, LightGBM, CatBoost) and Random Forest in the candidate set. Logistic Regression is retained as a baseline.

4. **Transaction amount and time carry interpretable signal.**
   Both features remain in the model unchanged. Tree splits handle their non-linear relationship with the target natively.

5. **Minor missingness (61 cells of ~7.5M).**
   Median imputation inside the sklearn Pipeline. The volume is too small to justify deletion-based handling, and median is robust to the heavy tails observed in the amount distribution.

6. **PCA components are uncorrelated.**
   No decorrelation step required. Linear models can use the features directly without ill-conditioning.

### Modeling decisions encoded in Notebook 02

| Decision               | Choice                                      | Justification                              |
|------------------------|---------------------------------------------|--------------------------------------------|
| Train/test split       | Stratified 70/30                            | Preserve fraud rate in both splits         |
| CV strategy            | Stratified 5-fold                           | Stable estimates under class imbalance     |
| Primary CV metric      | AUC-PR (average_precision)                  | Correct for imbalanced classification      |
| Imputation             | SimpleImputer(strategy="median")            | Robust to heavy tails                      |
| Scaling                | StandardScaler inside Pipeline              | Eliminates leakage; required by LR, SVM, MLP |
| Hyperparameter search  | Optuna TPE sampler, seed=42                 | Bayesian search; 10x more sample-efficient than random |
| Experiment tracking    | MLflow, project-local URI                   | Open source; no account required; reproducible across machines |
| Final model selection  | Minimum expected dollar cost                | Business-relevant metric                   |
| Production threshold   | 200-point cost sweep                        | Optimize the decision boundary in dollar units |